# Smarter way to do the inverse using a layer approach

Let's consider $f(x, \theta)$ a neural network where $k, n$ denote the batch and dimensionality of $x$ respectively. Let $p$ the number of total parameters and let $\theta_k$ be the parameters in each layer. For notational purposes, let $\theta_i$ just mean any arbitrary parameter. 
Suppose the network is of the sort 
$y_1 = L_1(x), y_2 = L_2(y_1), \ldots, y_N = L_N(y_{N-1})$. 

The key quantity we're investigating is the Levenberg-Marquardt matrix 
$$
(\lambda I + J^TJ)^{-1}
$$
where $J \in \mathbb R^{(k \times n) \times p} $ the Jacobian wrt to the parameters. Note that each entry of 
$$
J_{ij} = \frac{\partial f(x, \theta)_i}{\partial \theta_j}
$$
We assume a key fact now, which is that each parameter is only used on a single layer (aka no parameter sharing); most feed forward model satisfies this. Thus be simple chain rule, there exists a layer $k$ such that 
$$
J_{ij} = \frac{\partial f(x, \theta)_i}{\partial \theta_j} = \frac{\partial f(x, \theta)_i}{\partial y_k} \frac{\partial y_k}{\partial \theta_j}
$$

In particular, this is how backprop works in general, where $\frac{\partial f(x, \theta)_i}{\partial y_N(x, \theta)}\frac{\partial y_N(x, \theta)}{\partial y_{N-1}(x, \theta)} \cdots$ are propagated backwards. I'm unsure what the ML literature calls this though, which is annoying. 

But using this, it imposes even more structure; in particular note that $\frac{\partial y_k}{\partial \theta_j}$ is a local derivative, with non-zeros only on parameters of that layer. Furthermore, $\frac{\partial f(x, \theta)_i}{\partial y_k}$, in theory, can be extracted when doing a backprop. 

Let us see this with the code below. 

## Boiler plate 

Imports and definitions 

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
import numpy as np
import flax.linen as nn
from flax.training import train_state
import optax
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


In [ ]:
global_adjoints = []
class AdjointHook(nn.Module):
    """
    A wrapper module that applies a custom VJP to any given layer
    to capture its incoming gradient (adjoint) during the backward pass.
    """
    layer: nn.Module  # The layer to wrap

    @nn.compact
    def __call__(self, *args, **kwargs):
        # Define the function to which we'll attach the custom VJP
        @jax.custom_vjp
        def layer_with_hook(params, *args, **kwargs):
            return self.layer.apply(params, *args, **kwargs)

        def layer_fwd(params, *args, **kwargs):
            output = self.layer.apply(params, *args, **kwargs)
            return output, (params, args, kwargs)

        def layer_bwd(res, g):
            params, args, kwargs = res

            # Calculate the VJP of the original wrapped layer
            # This computes the gradients w.r.t. params and inputs
            _, vjp_fun = jax.vjp(
                lambda p, *a, **kw: self.layer.apply(p, *a, **kw), params, *args, **kwargs
            )
            
            # The VJP function returns a tuple of gradients
            grad_params, *grad_args = vjp_fun(g)

            # print(f"--- Captured Adjoint for layer: {self.layer.name} ---")
            # print(f'{g=} {grad_args[0]}')
            # print(f'{grad_params}')
            # print("-" * 30)
            global_adjoints.append(grad_args[0]
                                  )
            return (grad_params,) + tuple(grad_args)

        # Attach the custom forward and backward functions
        layer_with_hook.defvjp(layer_fwd, layer_bwd)
        
        # Get the parameters for the wrapped layer
        layer_params = self.param('wrapped_layer', self.layer.init, *args, **kwargs)

        return layer_with_hook(layer_params, *args, **kwargs)

class MLP(nn.Module):
    num_units: int
    def setup(self):
        self.dense1 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
        self.dense2 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
    def __call__(self, x):
        f = self.dense1(x)
        f = nn.tanh(f)
        f = self.dense2(f)
        return f 
    

class SimpleMLP(nn.Module):
    num_layers: int
    num_units: int
    num_classes: int

    def setup(self):
        # Create a list of Dense layers
        self.layers = (
            *[
                AdjointHook(MLP(self.num_units, name=f"layer_{i:02})")) for i in range(self.num_layers - 1)
            ], 
            AdjointHook(MLP(self.num_units, name=f"layer_{self.num_layers-1:02}")) # For now, do all the same dimensions 
        )
        
    def __call__(self, x):
        # Store input to match the notes
        # self.sow('intermediates',  f'layer_{0}_output', x)
        
        # Pass the input through each layer
        for i, layer in enumerate(self.layers):
            x = layer(x)
            
            # Store input to match the notes
            # self.sow('intermediates', f'layer_{i+1}_output', x)
        
        return x
        

## Create and store some model 

We create a simple model with 4 layers, with batch size of 2 and dimensions of 5.

In [ ]:
# Model definition 
L = 4
n = 5
n_samples = 2
lamb = 0.1
X = jax.random.normal(jax.random.PRNGKey(0), (n_samples, n))

key = jax.random.PRNGKey(0)

model = SimpleMLP(num_layers=L, num_units=n, num_classes=n)
params = model.init(key, jnp.ones((1, n)))

# Regular way
global_adjoints = []
jacobian = jax.jacobian(model.apply, argnums=0)(params, X)
print(global_adjoints[0].val.shape)

In [ ]:
model.a = 'test'

In [ ]:
model.a

## First, we construct the inverse exactly and store it so that we can compare 

In [ ]:
flattened, _ = jax.tree.flatten(
    jacobian
)

# Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
J = aggregated_array.reshape((n * n_samples, -1))



## LM matrix 

The GN/LM matrix has dimensions $p \times p$, and it's regularized inverse can be *quite diagonally dominant* if $\lambda \ll 1$. 

*This is why when $\lambda \ll 1$ that the steps become incredibly finicky: the scaling on the diagonal impilies that we are taking massive steps
$$
(\lambda I + J^TJ) p = \nabla_\theta \mathcal L
$$

From a math perspective this makes sense too, $J^TJ$ is relatively low-rank compared to $\lambda I$, meaning that the inverse scales as $1/\lambda$. 

In [ ]:
our_matrix_inv = jnp.linalg.inv(lamb * jnp.eye(J.shape[1]) +  J.T @ J)
our_matrix_inv

# Second try to find the matrix in a layer by layer way

Now, remember that $J^TJ$ is hard to compute in full since dense; we don't want to expand the full matrix and perform the matrix multiply, then do the solve. $p \times p$ is huge. 

## First, note the SMW formulation 

This is simply 
$$
(\lambda I+J^TJ)^{-1}=\frac1\lambda I−\frac{1}{\lambda^2} J^T(I+\frac{1}{\lambda} JJ^T)^{-1}J
$$
A key point here is that the forward products $J\vec v, J^T \vec w$ are actually easy, to compute and are part of the backprop anyways. Thus the difficulty lies in how do we compute the inner inverse. 

Let us verify that the two formulations are equal. 

In [ ]:
jnp.linalg.norm((1/lamb) * jnp.eye(J.shape[1]) - (1/(lamb**2)) * J.T @ \
    jnp.linalg.inv(jnp.eye(J.shape[0]) + (1/lamb) * J @ J.T) \
    @ J - our_matrix_inv)

## Layer formulation of Jacobian 

They key identity we want to exploit is stated above, which is 
$$
J_{ij} = \frac{\partial f(x, \theta)_i}{\partial \theta_j} = \frac{\partial f(x, \theta)_i}{\partial y_k} \frac{\partial y_k}{\partial \theta_j}
$$
meaning that
$$
J = M K
$$
where $M$ corresponds to all the adjoints $\frac{\partial f(x, \theta)_i}{\partial y_k}$ while $K$ is the Jacobian of *each* layer wrt to it's own parameters. 

$M$ as a matrix is actually quite small, with one dimension scaling as the number of layers, and the other as the adjoint variables, meaning it scales as the number of inputs by the number of outputs (e.g. if a layer is $y_k = L_k(y_{k-1})$ where $y_k, y_{k-1}$ have dimensions $a, b$ respectively, then the adjoint for that layer has $(k \times a) \times (k \times b)$. In my simple case, they're the same dimensions for simplicity. (Ooops wait, is this correct? Idea is correct)

Now, turning to the SMW formulation $JJ^T = MKK^TM^T$, an observation here is that $KK^T$ is now a small *block diagonal* matrix, with each block corresponding to each layer, meaning that $JJ^T$ as a whole can be done in a for loop for each layer instead of a monolith. 

We first grab the adjoints... and reformat it. This is the $M$ matrix.

In [ ]:
# I don't know how to store the adjoints in the model itself...
global_adjoints = []
jacobian = jax.jacobian(model.apply, argnums=0)(params, X)

global_adjoints = [adj.val.reshape(n * n_samples, n * n_samples) for adj in global_adjoints]
global_adjoints.insert(0, jnp.eye(global_adjoints[0].shape[0]))
global_adjoints.reverse() # Backward prop; so reverse 

In [ ]:
global_adjoints

## Now we need the gradients for each level

This is the $K$ matirx; we just save each block. Note that we rquire the intermediate outputs from each layer here. When doing the forward, we need to save this. 

In [ ]:
prediction, intermediates = model.apply(params, X, mutable=['intermediates'])
                                        
# Create layer model; then be able to take the gradients and stuff
layer = MLP(num_units=n)

# First define function
def apply_layer(params, x): 
    return layer.apply(params, x)

# This is relatively small I think... 
K_i = jax.jacrev(apply_layer, argnums=0)

k_list = []
for l in range(L): 
    flattened, _ = jax.tree.flatten(
            # Note the "intermediates" usage
            K_i(params['params'][f'layers_{l}']['wrapped_layer'], intermediates['intermediates'][f'layer_{l}_output'][0])
    )
    
    # Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
    reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
    aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1).reshape(n * n_samples, -1)
    k_list.append(aggregated_array)


In [ ]:
k_list

## Now, constructing the inner matrx of SMW 

This should be equal to $I + JJ^T/\lambda$, which we can verify. 

In [ ]:
# Size of (n \times k) \times (n \times k)
sum_mat = jnp.eye(J.shape[0]) 

# For each layer 
for i in range(L):
    # Each dgdt item is (n \times k) \times (p_i) where p_i 
    # is parameter of that layer 
    middle = k_list[i] @ k_list[i].T

    # Adjoints is really scaling as n * k**2
    sum_mat += global_adjoints[i+1] @ middle @ global_adjoints[i+1].T / lamb
print(
    jnp.linalg.norm(sum_mat - (jnp.eye(J.shape[0]) + J @ J.T/lamb))
)
sum_mat 

In [ ]:
# Using this small matrix to find the full matrix is cheap
(1/lamb) * jnp.eye(J.shape[1]) - 1 / lamb ** 2 * J.T @ jnp.linalg.inv(sum_mat) @ J

In [ ]:
jnp.linalg.norm((1/lamb) * jnp.eye(J.shape[1]) - 1 / lamb ** 2 * J.T @ jnp.linalg.inv(sum_mat) @ J - our_matrix_inv)

# This seems great right?

We can calculate the inverse exactly using by-products of backprop and a list of layers wrt to parameters (can be done in parallel;generally small, just need to store the intermediates). 
Total cost would be cost of applying vjp, jvp and the inversion of a $(k\times n) \times (k \times n)$ system which probably dominated by vjp/jvp. 

Well, there's a cheat I used. In particular, finding the **full** adjoints is not cheap. In recording the `global_adjoints` list, I actually called `jax.jacobian` which backprops the output of the network (in our case $k \times n$) back, meaning each parameter now takes up memory $k \times n$ times more space. 

We don't actually want to do this; in usual backprop, we only do a a `vjp` vector Jacobian product. A typical algorithm is something like:
1. Calculate residual $\vec w$
2. Backprop ($\vec w \cdot \frac{\partial f}{\partial \theta}$)

So why can't we extract the adjoint matrix from this by passing in say, an identity matrix or the sort?
It's because during a vjp calculation, it's grouped such that the dot product is computed at step one e.g.
$$
\left(\left(w^T \cdot \frac{\partial f(x, \theta)_i}{\partial y_N(x, \theta)} \right)\cdot\frac{\partial y_N(x, \theta)}{\partial y_{N-1}(x, \theta)}\right) \cdots
$$
which collapses the matrices we want.
We can extract out all the adjoints, but must pass in basis functions corresponding to each $n$, making this very inefficient. (" and the FLOP cost for evaluating vjp
is only about three times the cost of evaluating $f$"
See below for some code...

Thoughts on this: 
1. Definitely for scalars, this formulation would scale well.
2. For $n$ large, we might be able to skirt around this by passing in average, so that for each term, we get a sum? Might be able to use this as a preconditioner.

In [ ]:
global_adjoints = []
_, vjp_fun = jax.vjp(model.apply, params, X)

In [ ]:
# inp = jnp.zeros_like(X)
# inp = inp.at[:, 0].set(1)

inp = jnp.ones_like(X) / n
print(inp)
vjp_fun(inp)

In [ ]:
global_adjoints

In [ ]:
global_adjoints[0]

In [ ]:
new_global_adjoints = []
for adj in global_adjoints: 
    stacked = jnp.repeat(adj, repeats=jnp.array([n, n]), axis=0)
    blocked = jnp.zeros((n * n_samples, n * n_samples), dtype=stacked.dtype)

    # A bit weird for now 
    blocked = blocked.at[0:5, 0:5].set(stacked[0:5, :])
    blocked = blocked.at[5:10, 5:10].set(stacked[5:, :])
    new_global_adjoints.append(blocked)


new_global_adjoints.insert(0, jnp.eye(new_global_adjoints[0].shape[0]))
new_global_adjoints.reverse() # Backward prop; so reverse 

new_global_adjoints



In [ ]:
# Size of (n \times k) \times (n \times k)
sum_mat = jnp.eye(J.shape[0]) 

# For each layer 
for i in range(L):
    # Each dgdt item is (n \times k) \times (p_i) where p_i 
    # is parameter of that layer 
    middle = k_list[i] @ k_list[i].T

    # Adjoints is really scaling as n * k**2
    sum_mat += new_global_adjoints[i+1] @ middle @ new_global_adjoints[i+1].T / lamb
print(
    jnp.linalg.norm(sum_mat - (jnp.eye(J.shape[0]) + J @ J.T/lamb))
)
jnp.linalg.eigvalsh(sum_mat)

In [ ]:
(jnp.eye(J.shape[0]) + J @ J.T/lamb)

In [ ]:
jnp.linalg.eigvalsh(jnp.eye(J.shape[0]) + J @ J.T/lamb)